In [4]:
import numpy as np
from itertools import combinations
from skimage.metrics import structural_similarity as compare_ssim
import matplotlib.pyplot as plt
import torch
from PIL import Image
import pickle
import sys
from tqdm import tqdm
import csv
from matplotlib.colors import Normalize, LogNorm
import scipy.ndimage as ndimage
sys.path.append('..')
import models_mae_gray
import pandas as pd
sys.path.append('bigclassroom')

# Function to save loss and flattened mask to CSV file
def save_loss_mask_to_csv(max_error, mean_error, min_error, mask, output_file):
    with open(output_file, 'a', newline='') as csvfile:
        fieldnames = ['max', 'mean', 'min_nonzero', 'custom_mask']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Write header only if file is empty
        if csvfile.tell() == 0:
            writer.writeheader()

        writer.writerow({
            'max': max_error,
            'mean': mean_error,
            'min_nonzero': min_error,
            'custom_mask': ','.join(map(str, mask.flatten()))
        })

def prepare_model(chkpt_dir, arch='mae_vit_large_patch56'):
    # Build model and load checkpoint
    model = getattr(models_mae_gray, arch)()
    checkpoint = torch.load(chkpt_dir, map_location='cpu')
    msg = model.load_state_dict(checkpoint['model'], strict=False)
    print(msg)
    return model

def inverse_normalize(log_normalized_values):
    # 將 0~1 的正規化數據反轉回 log 範圍，然後再轉回線性空間的原始範圍
    original_min = 300
    original_max = 1600
    log_values = log_normalized_values * (np.log(original_max) - np.log(original_min)) + np.log(original_min)
    original_values = np.exp(log_values)
    return original_values

def run_one_image(img, x_original, model, mask_ratio, custom_mask=None):
    img = img.values.astype('float32')
    x = torch.tensor(img).unsqueeze(0).unsqueeze(1)

    # Run MAE
    loss, y, mask = model(x.float(), mask_ratio=mask_ratio, custom_mask=custom_mask)

    y = model.unpatchify(y).detach().cpu()

    # Visualize mask
    mask = mask.detach()
    mask = mask.unsqueeze(-1).repeat(1, 1, model.patch_embed.patch_size[0]**2)
    mask = model.unpatchify(mask).detach().cpu()

    # Masked image
    im_masked = x * (1 - mask)

    # MAE reconstructed image
    im_paste = x * (1 - mask) + y * mask

    # Convert tensors to numpy arrays for visualization (before log transformation)
    x_original_np = x.squeeze().detach().cpu().numpy()
    im_masked_np = im_masked.squeeze().detach().cpu().numpy()
    im_paste_np = im_paste.squeeze().detach().cpu().numpy()

    # Directly set the masked areas to gray in im_masked_np
    im_masked_np[mask.squeeze().cpu().numpy() == 1] = 0.5  # 灰色值為 0.5

    # Flatten images for inverse transformation
    im_paste_flat = im_paste.flatten().detach().numpy()
    x_flat = x.flatten().detach().numpy()

    # Inverse normalization to recover original values
    im_paste_original = inverse_normalize(im_paste_flat)
    x_original = inverse_normalize(x_flat)

    # Reshape to original dimensions
    x_original = x_original.reshape(4, 4)
    im_paste_original = im_paste_original.reshape(4, 4)


    # 計算相對誤差並打印
    relative_error = np.abs((x_original - im_paste_original) / x_original)
    max_error = np.max(relative_error)

    # 只考虑非零相对误差来计算最小值和平均值
    non_zero_relative_error = relative_error[relative_error > 0]
    min_error = np.min(non_zero_relative_error) if non_zero_relative_error.size > 0 else 0
    mean_error = np.mean(non_zero_relative_error) if non_zero_relative_error.size > 0 else 0

    # Return errors for further analysis
    return max_error, mean_error, min_error

from concurrent.futures import ProcessPoolExecutor

def process_mask_combination(comb, fixed_positions, size):
    # Initialize all positions with 1
    mask = np.ones((size, size), dtype=int)
    
    # Only set positions if fixed_positions is not empty
    if fixed_positions:
        mask[np.unravel_index(np.array(fixed_positions, dtype=np.intp), (size, size))] = 0
    
    # Set selected positions to 0
    mask[np.unravel_index(comb, (size, size))] = 0
    return mask

def generate_custom_masks(size, img, model, x_original):
    all_positions = list(range(size*size))
    fixed_positions = []  # Empty list for no fixed positions
    num_zeros = 4  # Set the number of zeros in the mask
    all_combinations = list(combinations(all_positions, num_zeros))
    
    results = []
    with ProcessPoolExecutor() as executor:
        futures = [executor.submit(process_mask_combination, comb, fixed_positions, size) for comb in all_combinations]
        for future in tqdm(futures, desc='Processing combinations'):
            mask = future.result()
            max_error, mean_error, min_error = run_one_image(
                img=img, 
                x_original=x_original, 
                model=model, 
                mask_ratio=0.75, 
                custom_mask=torch.tensor(mask, dtype=torch.int)
            )
            save_loss_mask_to_csv(max_error, mean_error, min_error, mask, './losses_masks.csv')
            results.append((max_error, mean_error, min_error, mask))

def conditional_interpolate(values):
    center = values[len(values) // 2]
    if np.isnan(center):
        neighbors = np.concatenate([values[:len(values) // 2], values[len(values) // 2 + 1:]])
        valid_neighbors = neighbors[~np.isnan(neighbors)]
        if len(valid_neighbors) > 0:
            return np.nanmean(valid_neighbors)
        else:
            return 0  # Return 0 to avoid NaN confusion
    else:
        return center

# chkpt_dir = 'fundamental_output_dir/checkpoint-9999.pth'

# chkpt_dir = 'finetune_hv-hc_output_dir/checkpoint-999.pth'
# chkpt_dir = 'finetune_all-hv-hc_output_dir/checkpoint-2999.pth'
# chkpt_dir = 'cfd_output_dir/checkpoint-6600.pth'
# chkpt_dir = 'finetune_hv-hc-5-sample-rate_output_dir/checkpoint-4999.pth'
# chkpt_dir = 'finetune_hv-hc-15-sample-rate_output_dir/checkpoint-4999.pth'
chkpt_dir = 'finetune_hv-hc-30-sample-rate_output_dir/checkpoint-4999.pth'
# chkpt_dir = 'finetune_hv-hc-60-sample-rate_output_dir/checkpoint-4999.pth'

model_mae_gray = prepare_model(chkpt_dir, 'mae_vit_base_patch1')
print('Model loaded.')

# Load data for 3330.csv, 3430.csv, and 3530.csv
# data = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_1000.csv', header=None, index_col=False)

data = pd.read_csv('cfd_hv-hc_dataset/exp_test/1111-1in3out-hv-hc-mid_3000.csv', header=None, index_col=False)

# Generate custom masks and save the results in CSV
generate_custom_masks(4, data, model_mae_gray, data)


<All keys matched successfully>
Model loaded.


Processing combinations: 100%|██████████| 1820/1820 [00:42<00:00, 43.21it/s] 


In [28]:
import numpy as np
from itertools import combinations
from skimage.metrics import structural_similarity as compare_ssim
import matplotlib.pyplot as plt
import torch
from PIL import Image
import pickle
import sys
from tqdm import tqdm
import csv
from matplotlib.colors import Normalize, LogNorm
import scipy.ndimage as ndimage
sys.path.append('..')
import models_mae_gray
import pandas as pd
sys.path.append('bigclassroom')

# Function to save loss and flattened mask to CSV file
def save_loss_mask_to_csv(max_error, mean_error, min_error, mask, output_file):
    with open(output_file, 'a', newline='') as csvfile:
        fieldnames = ['max', 'mean', 'min_nonzero', 'custom_mask']
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames)

        # Write header only if file is empty
        if csvfile.tell() == 0:
            writer.writeheader()

        writer.writerow({
            'max': max_error,
            'mean': mean_error,
            'min_nonzero': min_error,
            'custom_mask': ','.join(map(str, mask.flatten()))
        })

def prepare_model(chkpt_dir, arch='mae_vit_large_patch56'):
    # Build model and load checkpoint
    model = getattr(models_mae_gray, arch)()
    checkpoint = torch.load(chkpt_dir, map_location='cpu')
    msg = model.load_state_dict(checkpoint['model'], strict=False)
    print(msg)
    return model

def inverse_normalize(log_normalized_values):
    # 將 0~1 的正規化數據反轉回 log 範圍，然後再轉回線性空間的原始範圍
    original_min = 300
    original_max = 1600
    log_values = log_normalized_values * (np.log(original_max) - np.log(original_min)) + np.log(original_min)
    original_values = np.exp(log_values)
    return original_values

def run_one_image(img, x_original, model, mask_ratio, custom_mask=None):
    img = img.values.astype('float32')
    x = torch.tensor(img).unsqueeze(0).unsqueeze(1)

    # Run MAE
    loss, y, mask = model(x.float(), mask_ratio=mask_ratio, custom_mask=custom_mask)

    y = model.unpatchify(y).detach().cpu()

    # Visualize mask
    mask = mask.detach()
    mask = mask.unsqueeze(-1).repeat(1, 1, model.patch_embed.patch_size[0]**2)
    mask = model.unpatchify(mask).detach().cpu()

    # Masked image
    im_masked = x * (1 - mask)

    # MAE reconstructed image
    im_paste = x * (1 - mask) + y * mask

    # Convert tensors to numpy arrays for visualization (before log transformation)
    x_original_np = x.squeeze().detach().cpu().numpy()
    im_masked_np = im_masked.squeeze().detach().cpu().numpy()
    im_paste_np = im_paste.squeeze().detach().cpu().numpy()

    # Directly set the masked areas to gray in im_masked_np
    im_masked_np[mask.squeeze().cpu().numpy() == 1] = 0.5  # 灰色值為 0.5

    # Flatten images for inverse transformation
    im_paste_flat = im_paste.flatten().detach().numpy()
    x_flat = x.flatten().detach().numpy()

    # Inverse normalization to recover original values
    im_paste_original = inverse_normalize(im_paste_flat)
    x_original = inverse_normalize(x_flat)

    # Reshape to original dimensions
    x_original = x_original.reshape(4, 4)
    im_paste_original = im_paste_original.reshape(4, 4)

    # 計算相對誤差並打印
    relative_error = np.abs((x_original - im_paste_original) / x_original)
    max_error = np.max(relative_error)

    # 只考慮非零相對誤差來計算最小值和平均值
    non_zero_relative_error = relative_error[relative_error > 0]
    min_error = np.min(non_zero_relative_error) if non_zero_relative_error.size > 0 else 0
    mean_error = np.mean(non_zero_relative_error) if non_zero_relative_error.size > 0 else 0

    # Return errors for further analysis
    return max_error, mean_error, min_error

def process_mask_combination(comb, fixed_positions, size):
    # Initialize all positions with 1
    mask = np.ones((size, size), dtype=int)
    
    # Only set positions if fixed_positions is not empty
    if fixed_positions:
        mask[np.unravel_index(np.array(fixed_positions, dtype=np.intp), (size, size))] = 0
    
    # Set selected positions to 0
    mask[np.unravel_index(comb, (size, size))] = 0
    return mask

def generate_custom_masks(size, img_list, model):
    all_positions = list(range(size*size))
    fixed_positions = []  # Empty list for no fixed positions
    num_zeros = 8 # Set the number of zeros in the mask
    all_combinations = list(combinations(all_positions, num_zeros))
    
    for comb in tqdm(all_combinations, desc='Processing combinations'):
        mask = process_mask_combination(comb, fixed_positions, size)
        mask_tensor = torch.tensor(mask, dtype=torch.int)
        
        # Calculate for each image in the list
        errors = [
            run_one_image(img=img, x_original=img, model=model, mask_ratio=0.75, custom_mask=mask_tensor)
            for img in img_list
        ]
        
        # Check if all max_errors are less than 0.1
        if all(max_error < 0.1 for max_error, _, _ in errors):
            # Save if the condition is met
            mean_errors = [mean_error for _, mean_error, _ in errors]
            min_errors = [min_error for _, _, min_error in errors]
            save_loss_mask_to_csv(
                max_error=errors[0][0],  # 第一個索引的 max_error
                mean_error=errors[0][1],  # 第一個索引的 mean_error
                min_error=errors[0][2],  # 第一個索引的 min_error
                mask=mask,
                output_file='./losses_masks_8.csv'
            )

def conditional_interpolate(values):
    center = values[len(values) // 2]
    if np.isnan(center):
        neighbors = np.concatenate([values[:len(values) // 2], values[len(values) // 2 + 1:]])
        valid_neighbors = neighbors[~np.isnan(neighbors)]
        if len(valid_neighbors) > 0:
            return np.nanmean(valid_neighbors)
        else:
            return 0  # Return 0 to avoid NaN confusion
    else:
        return center

# Load model and data
# chkpt_dir = '1111-1in3out-hv-lc-mid_output_dir/checkpoint-9999.pth'
chkpt_dir = 'fundamental_output_dir/checkpoint-9999.pth'

model_mae_gray = prepare_model(chkpt_dir, 'mae_vit_base_patch1')
print('Model loaded.')

# Load data for 3330.csv, 3430.csv, and 3530.csv
data_1 = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_1000.csv', header=None, index_col=False)
data_2 = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_2000.csv', header=None, index_col=False)
data_3 = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_3000.csv', header=None, index_col=False)

# data_4 = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_2800.csv', header=None, index_col=False)
# data_5 = pd.read_csv('fundamental_dataset/test/1119-1in3out-mv-mc-mid_3500.csv', header=None, index_col=False)
# Generate custom masks and save the results in CSV
generate_custom_masks(4, [data_1, data_2, data_3], model_mae_gray)



<All keys matched successfully>
Model loaded.


Processing combinations: 100%|██████████| 12870/12870 [13:44<00:00, 15.61it/s]
